# vrgrid — FRNet head+backbone fine-tune

The two configurations the laptop **cannot run**. On its RTX 5050 both die with

    torch.OutOfMemoryError: GPU 0 has a total capacity of 7.57 GiB
                            of which 102.19 MiB is free

Head-only fine-tuning trains 0.9% of parameters and fits in 8 GB. Unfreezing the
backbone means holding activations for the whole stack, and it does not. The T4's
15.6 GB is roughly double, so this is the one piece of work here that needs a
**larger** card rather than a faster one.

The question: does unfreezing the backbone beat the pretrained **65.2% mIoU**,
where head-only only ever drew level with it?

Scored by `frnet_eval.py` on seq 08, 200 frames — the same slice as every other
row, and a sequence `frnet_finetune.py` refuses to train on.

In [ ]:
# 1. The machine. This is the whole point of the notebook -- a DIFFERENT card.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU -- set Accelerator to GPU in the sidebar"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {vram:.1f} GB")
assert vram > 12, f"{name} has {vram:.1f} GB -- the 16 GB contention test needs more than the laptop's 8"

In [ ]:
# 2. The repo. Public, so no token.
%cd /kaggle/working
!rm -rf vrgrid-26
!git clone -q https://github.com/Stxtics03/vrgrid-26.git
%cd /kaggle/working/vrgrid-26
!git log --oneline -1
!pip -q install -e . 2>&1 | tail -2
try:
    import cupy; print("cupy", cupy.__version__, "(preinstalled)")
except ImportError:
    !pip -q install cupy-cuda12x

In [ ]:
# 4. Assemble the asset tree vrgrid expects. The mirror supplies sequences/; the
#    bundle supplies poses/ and the checkpoint.
#    loader.py uses the OFFICIAL KITTI GT poses at poses/<seq>.txt, NOT the
#    SemanticKITTI SLAM poses inside sequences/<seq>/poses.txt. Do not substitute
#    one for the other -- they are different quantities and the swap is silent.
import os
from pathlib import Path

# Print what actually mounted before asserting anything. A dataset that is
# attached but empty looks identical to one that is missing, three cells later.
IN = Path("/kaggle/input")
for root, dirs, files in os.walk(IN):
    depth = root.replace(str(IN), "").count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print("  " * depth + os.path.basename(root) + "/")

def find(child, maxdepth=4):
    """The mounted dataset directory containing <child>.

    Kaggle nests these as /kaggle/input/datasets/<owner>/<slug>/, so a
    top-level scan finds only "datasets". Breadth-first with a depth cap --
    never rglob, the mirror alone is 96 GB and 66k files.
    """
    frontier = [IN]
    for _ in range(maxdepth):
        nxt = []
        for d in frontier:
            if (d / child).is_dir():
                return d
            try:
                nxt += [k for k in d.iterdir() if k.is_dir()]
            except (PermissionError, OSError):
                pass
        frontier = nxt
    return None

MIRROR = find("sequences")
BUNDLE = find("poses")
print("\nMIRROR:", MIRROR, "\nBUNDLE:", BUNDLE)
assert MIRROR, "no mounted dataset contains sequences/"
assert BUNDLE, "no mounted dataset contains poses/ -- check the Input panel"

A = Path("/kaggle/working/assets")
(A / "dataset").mkdir(parents=True, exist_ok=True)
for src, dst in [(MIRROR / "sequences", A / "dataset/sequences"),
                 (BUNDLE / "poses",     A / "dataset/poses"),
                 (BUNDLE / "checkpoints", A / "checkpoints")]:
    assert src.exists(), f"missing: {src}"
    if not dst.exists():
        dst.symlink_to(src)

os.environ["VRGRID_ASSETS"] = str(A)
os.environ["VRGRID_DATA_ROOT"] = str(A / "dataset")
os.environ["VRGRID_FRNET_CHECKPOINT"] = str(A / "checkpoints/frnet-semantickitti_seg.pth")
for k in ("VRGRID_ASSETS", "VRGRID_DATA_ROOT", "VRGRID_FRNET_CHECKPOINT"):
    print(f"{k}={os.environ[k]}")


In [ ]:
# Give the allocator room to breathe before anything big is built. The laptop
# failed asking for 246 MiB with 102 MiB free -- that is fragmentation as much
# as capacity, and expandable_segments is the documented fix.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, pathlib, subprocess, sys
OUT = pathlib.Path("/kaggle/working/backbone"); OUT.mkdir(exist_ok=True)
CK  = pathlib.Path("/kaggle/working/ckpt");     CK.mkdir(exist_ok=True)
free, total = torch.cuda.mem_get_info()
print(f"VRAM free {free/1e9:.1f} GB of {total/1e9:.1f} GB "
      f"(the laptop had 7.57 GB total and died)")

In [ ]:
# 1. The config that OOM'd. Backbone at a low LR is the standard fine-tune.
NAME, ARGS = "backbone-lowlr", "--steps 2000 --unfreeze head+backbone --lr 1e-4"
rc = subprocess.run(
    f"python scripts/frnet_finetune.py --fast-scatter {ARGS} "
    f"--out {CK}/{NAME}.pth 2>&1 | tee {OUT}/{NAME}.train.log",
    shell=True, cwd="/kaggle/working/vrgrid-26").returncode
if rc == 0:
    subprocess.run(
        f"python scripts/frnet_eval.py --seq 08 --frames 200 --fast-scatter "
        f"--checkpoint {CK}/{NAME}.pth 2>&1 | tee {OUT}/{NAME}.eval.log",
        shell=True, cwd="/kaggle/working/vrgrid-26")
else:
    print(f"{NAME} FAILED rc={rc} -- see {OUT}/{NAME}.train.log")
    !tail -20 {OUT}/{NAME}.train.log
torch.cuda.empty_cache()

In [ ]:
# 2. Same, minus the 3x class weighting -- the thing that cost the laptop run its mIoU.
NAME, ARGS = "backbone-noclsw", "--steps 2000 --unfreeze head+backbone --lr 1e-4 --weight-classes ''"
rc = subprocess.run(
    f"python scripts/frnet_finetune.py --fast-scatter {ARGS} "
    f"--out {CK}/{NAME}.pth 2>&1 | tee {OUT}/{NAME}.train.log",
    shell=True, cwd="/kaggle/working/vrgrid-26").returncode
if rc == 0:
    subprocess.run(
        f"python scripts/frnet_eval.py --seq 08 --frames 200 --fast-scatter "
        f"--checkpoint {CK}/{NAME}.pth 2>&1 | tee {OUT}/{NAME}.eval.log",
        shell=True, cwd="/kaggle/working/vrgrid-26")
else:
    print(f"{NAME} FAILED rc={rc} -- see {OUT}/{NAME}.train.log")
    !tail -20 {OUT}/{NAME}.train.log
torch.cuda.empty_cache()

In [ ]:
# 3. Everything unfrozen, LR lower again. Only reachable with 15.6 GB.
NAME, ARGS = "all-lowlr", "--steps 2000 --unfreeze all --lr 5e-5 --weight-classes ''"
rc = subprocess.run(
    f"python scripts/frnet_finetune.py --fast-scatter {ARGS} "
    f"--out {CK}/{NAME}.pth 2>&1 | tee {OUT}/{NAME}.train.log",
    shell=True, cwd="/kaggle/working/vrgrid-26").returncode
if rc == 0:
    subprocess.run(
        f"python scripts/frnet_eval.py --seq 08 --frames 200 --fast-scatter "
        f"--checkpoint {CK}/{NAME}.pth 2>&1 | tee {OUT}/{NAME}.eval.log",
        shell=True, cwd="/kaggle/working/vrgrid-26")
else:
    print(f"{NAME} FAILED rc={rc} -- see {OUT}/{NAME}.train.log")
    !tail -20 {OUT}/{NAME}.train.log
torch.cuda.empty_cache()

In [ ]:
# Collect. Download backbone-results.zip and drop it into docs/gpu-lane/t4/.
!cd /kaggle/working && zip -qr backbone-results.zip backbone && ls -la backbone/
print()
for f in sorted(OUT.glob("*.eval.log")):
    print("==", f.stem, "==")
    for line in f.read_text().splitlines():
        if any(k in line for k in ("point accuracy", "mIoU over", "drivable")):
            print("  ", line.strip())